# StormEngine V7-B workflow

390-point baseline: 239 physical DPC stations plus 151 model-derived Open-Meteo marine support points. Run one stage at a time.

In [ ]:
from pathlib import Path
import subprocess, sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
CONFIG='configs/v7_b.yaml'
DEVICE='cuda'
RUN_PILOT=False
RUN_FORMAL_TRAINING=False
RUN_2017_EVALUATION=False
RUN_COMBINED_REPLAY=False
SAVE_PREDICTIONS=False
def run(*args):
    command=[sys.executable,*map(str,args)]
    print(' '.join(command),flush=True)
    process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
    assert process.stdout is not None
    for line in process.stdout: print(line,end='',flush=True)
    code=process.wait()
    if code: raise subprocess.CalledProcessError(code,command)
print(ROOT)

## 1. Unit tests, ERA5 preflight, and real 390-point input check

In [ ]:
run('-m','pytest','tests/test_v7_dataset.py','tests/test_v7_input.py','tests/test_v7_model.py','tests/test_open_meteo.py','-q')
run('scripts/check_v7.py','preflight','--config',CONFIG,'--device',DEVICE)
run('scripts/check_v7_b_input.py')

## 2. Smoke test and 200-batch benchmark

In [ ]:
run('scripts/check_v7.py','smoke','--config',CONFIG,'--device',DEVICE)
run('scripts/check_v7.py','benchmark','--config',CONFIG,'--device',DEVICE,'--batches','200')

## 3. Short pilot
Enable only after smoke and benchmark pass.

In [ ]:
if RUN_PILOT:
    run('scripts/check_v7.py','pilot','--config',CONFIG,'--device',DEVICE,'--epochs','5')
else: print('Pilot skipped. Set RUN_PILOT=True when ready.')

## 4. Formal 2010-2015 training
Enable only after reviewing pilot. Uses 2016 for model selection and saves best.pt/last.pt.

In [ ]:
if RUN_FORMAL_TRAINING:
    run('scripts/check_v7.py','train','--config',CONFIG,'--device',DEVICE)
else: print('Formal training skipped.')

## 5. Frozen 2017 evaluation

In [ ]:
if RUN_2017_EVALUATION:
    run('scripts/evaluate_v7_a.py','--config',CONFIG,'--checkpoint','artifacts/v7_b_2010_2017/best.pt','--scenario','clean','--device',DEVICE,'--output-dir','artifacts/v7_b_2010_2017/evaluation_2017_clean_seed42')
    for seed in (42,123,2026): run('scripts/evaluate_v7_a.py','--config',CONFIG,'--checkpoint','artifacts/v7_b_2010_2017/best.pt','--scenario','missing','--seed',str(seed),'--device',DEVICE,'--output-dir',f'artifacts/v7_b_2010_2017/evaluation_2017_missing_seed{seed}')
else: print('2017 evaluation skipped.')

## 6. Real combined DPC + Open-Meteo replay
Uses only the past 12 valid times from each source. Open-Meteo is model-derived support, not an observation.

In [ ]:
if RUN_COMBINED_REPLAY:
    args=['scripts/replay_v7_b.py','--config',CONFIG,'--device',DEVICE]
    if SAVE_PREDICTIONS: args.append('--save-predictions')
    run(*args)
else: print('Combined replay skipped.')